# 08 - Data Quality Gate (job task)
Runs data quality checks across Bronze, Silver and Gold.
1. Every result is appended to `gold.data_quality_results` (audit trail per run)
2. If ANY check fails, the notebook raises an error -> the job fails -> email alert

**Job parameters:** `min_fact_rows` (threshold), `job_run_id` (passed from the job for traceability)

In [0]:
dbutils.widgets.text("min_fact_rows", "100000")
dbutils.widgets.text("job_run_id", "manual")

min_fact_rows = int(dbutils.widgets.get("min_fact_rows"))
job_run_id = dbutils.widgets.get("job_run_id")
C = "retail_lakehouse"

print(f"min_fact_rows = {min_fact_rows:,} | job_run_id = {job_run_id}")

In [0]:
import os
import sys

# Repo's src/ folder is one level up from notebooks/
sys.path.append(os.path.abspath("../src"))
from retail_lakehouse.quality import build_checks, evaluate

results = []
for name, layer, sql, op, expected in build_checks(C, min_fact_rows):
    raw = spark.sql(sql).first()[0]
    actual = int(raw) if raw is not None else None
    passed = evaluate(actual, op, expected)
    results.append((job_run_id, name, layer, actual if actual is not None else -1, f"{op} {expected}", passed))
    shown = f"{actual:,}" if actual is not None else "NULL"
    print(f"{'✅' if passed else '❌'} {name:35} actual={shown:>10}  expected {op} {expected:,}")

In [0]:
from pyspark.sql import functions as F

results_df = spark.createDataFrame(
    results,
    "job_run_id STRING, check_name STRING, layer STRING, actual_value BIGINT, rule STRING, passed BOOLEAN",
).withColumn("checked_at", F.current_timestamp())

results_df.write.mode("append").saveAsTable(f"{C}.gold.data_quality_results")
display(results_df)

failed = [r[1] for r in results if not r[5]]
if failed:
    raise Exception(f"❌ DATA QUALITY GATE FAILED: {len(failed)} check(s): {', '.join(failed)}")

print(f"✅ All {len(results)} data quality checks passed")

In [0]:
%sql
SELECT job_run_id,
       MIN(checked_at)                                  AS checked_at,
       COUNT(*)                                         AS checks_run,
       SUM(CASE WHEN passed THEN 0 ELSE 1 END)          AS checks_failed,
       concat_ws(', ', collect_list(CASE WHEN NOT passed THEN check_name END)) AS failed_checks
FROM retail_lakehouse.gold.data_quality_results
GROUP BY job_run_id
ORDER BY checked_at DESC;